# Databricks Outbound Activity Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Data Exfiltration Detection
> **Data Sources:** DatabricksNotebook, DatabricksSecrets, DatabricksDBFS, DatabricksClusters, DatabricksJobs, DatabricksSQLPermissions, IdentityInfo, AADUserRiskEvents, BehaviorAnalytics
> **Nodes:** 3 (User, Resource, Capability)
> **Edges:** 3 (ACCESSED, ResourceHasCapability, UserHasCapability)

Maps Databricks notebook and cluster activity to user identities, enriched with Entra ID risk signals and UEBA behavioral analytics. Enables detection of unusual outbound data movement, privilege escalation through cluster access, and data exfiltration patterns.

## Environment & Configuration

Import required packages and configure the Sentinel Data Lake connection.


In [ ]:
# =============================================================================
# Databricks Outbound Activity Graph # =============================================================================
# CONFIGURATION: Update the workspace_name below before running
# =============================================================================

from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
 col, lit, coalesce, get_json_object, concat_ws,
 to_date, countDistinct, count, min, max, first, lower, trim
)
from datetime import datetime, timedelta

# =============================================================================
# ⚠ CUSTOMER CONFIGURATION - UPDATE THESE VALUES
# =============================================================================
# ⚠ REQUIRED: Set your Microsoft Sentinel workspace name before running this notebook
workspace_name = "<YOUR_WORKSPACE_NAME>" # ← Replace with your Sentinel workspace name
# 📅 OPTIONAL: Adjust the lookback window (default: 7 days). Use 1 for IR, 14 for hunting, 30+ for historical.
LOOKBACK_DAYS = 1
LOOKBACK_DATE = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
# =============================================================================

# Initialize the Sentinel Lake provider
lake_provider = MicrosoftSentinelProvider(spark)

print(f" Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Lookback from: {LOOKBACK_DATE}")
print(f"🏢 Workspace: {workspace_name}")
print(f"📋 ")

if workspace_name == "YOUR_WORKSPACE_NAME":
 print("\n⚠ WARNING: Please update 'workspace_name' with your actual Sentinel workspace name!")
 print(" Example: workspace_name = \"contoso-sentinel-prod\"")


## Data Ingestion

Read source tables from the Sentinel Data Lake, apply time and quality filters, and persist to Spark memory.


In [ ]:
# =============================================================================
# STEP 1: Load and PERSIST raw tables
# =============================================================================
# Primary Data: 6 Databricks Audit Tables
# Enrichment Data: IdentityInfo, AADUserRiskEvents, BehaviorAnalytics
# =============================================================================

# ===================== PRIMARY: DATABRICKS AUDIT TABLES =====================
raw_notebook = lake_provider.read_table("DatabricksNotebook", workspace_name)
raw_notebook = raw_notebook.df if hasattr(raw_notebook, 'df') else raw_notebook
raw_notebook = raw_notebook.persist()

raw_secrets = lake_provider.read_table("DatabricksSecrets", workspace_name)
raw_secrets = raw_secrets.df if hasattr(raw_secrets, 'df') else raw_secrets
raw_secrets = raw_secrets.persist()

raw_dbfs = lake_provider.read_table("DatabricksDBFS", workspace_name)
raw_dbfs = raw_dbfs.df if hasattr(raw_dbfs, 'df') else raw_dbfs
raw_dbfs = raw_dbfs.persist()

raw_clusters = lake_provider.read_table("DatabricksClusters", workspace_name)
raw_clusters = raw_clusters.df if hasattr(raw_clusters, 'df') else raw_clusters
raw_clusters = raw_clusters.persist()

raw_jobs = lake_provider.read_table("DatabricksJobs", workspace_name)
raw_jobs = raw_jobs.df if hasattr(raw_jobs, 'df') else raw_jobs
raw_jobs = raw_jobs.persist()

raw_permissions = lake_provider.read_table("DatabricksSQLPermissions", workspace_name)
raw_permissions = raw_permissions.df if hasattr(raw_permissions, 'df') else raw_permissions
raw_permissions = raw_permissions.persist()

print(f" Databricks Audit Tables loaded:")
print(f"📊 DatabricksNotebook: {raw_notebook.count()}")
print(f"📊 DatabricksSecrets: {raw_secrets.count()}")
print(f"📊 DatabricksDBFS: {raw_dbfs.count()}")
print(f"📊 DatabricksClusters: {raw_clusters.count()}")
print(f"📊 DatabricksJobs: {raw_jobs.count()}")
print(f"📊 DatabricksSQLPermissions: {raw_permissions.count()}")

# ===================== ENRICHMENT: IDENTITY & RISK TABLES =====================
raw_identity = lake_provider.read_table("IdentityInfo", workspace_name)
raw_identity = raw_identity.df if hasattr(raw_identity, 'df') else raw_identity
raw_identity = raw_identity.persist()

raw_risk_events = lake_provider.read_table("AADUserRiskEvents", workspace_name)
raw_risk_events = raw_risk_events.df if hasattr(raw_risk_events, 'df') else raw_risk_events
raw_risk_events = raw_risk_events.persist()

raw_behavior = lake_provider.read_table("BehaviorAnalytics", workspace_name)
raw_behavior = raw_behavior.df if hasattr(raw_behavior, 'df') else raw_behavior
raw_behavior = raw_behavior.persist()

print(f"\n Enrichment Tables loaded:")
print(f"📊 IdentityInfo: {raw_identity.count()}")
print(f"📊 AADUserRiskEvents: {raw_risk_events.count()}")
print(f"📊 BehaviorAnalytics: {raw_behavior.count()}")


In [ ]:
# =============================================================================
# STEP 2: Parse ALL fields ONCE into unified outbound_df
# =============================================================================

from pyspark.sql.functions import when

# Helper: Add common fields
def parse_common_fields(df):
 return df.withColumn("UserEmail", get_json_object(col("Identity"), "$.email"))

# Parse DatabricksNotebook
notebook_df = (raw_notebook
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("ActionName").isin(["downloadNotebook", "exportNotebook"]))
 .transform(parse_common_fields)
 .withColumn("ResourceId", get_json_object(col("RequestParams"), "$.path"))
 .withColumn("ResourceType", lit("Notebook"))
 .withColumn("CapabilityId", lit("CAN_DOWNLOAD_NOTEBOOKS"))
 .select("UserEmail", "ResourceId", "ResourceType", "CapabilityId", 
 "ActionName", "TimeGenerated", "SourceIPAddress")
)

# Parse DatabricksSecrets
secrets_df = (raw_secrets
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("ActionName") == "getSecret")
 .transform(parse_common_fields)
 .withColumn("ResourceId", concat_ws("/", 
 get_json_object(col("RequestParams"), "$.scope"),
 get_json_object(col("RequestParams"), "$.key")))
 .withColumn("ResourceType", lit("Secret"))
 .withColumn("CapabilityId", lit("CAN_ACCESS_SECRETS"))
 .select("UserEmail", "ResourceId", "ResourceType", "CapabilityId", 
 "ActionName", "TimeGenerated", "SourceIPAddress")
)

# Parse DatabricksDBFS
dbfs_df = (raw_dbfs
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("ActionName").isin(["put", "create", "mkdirs"]))
 .transform(parse_common_fields)
 .withColumn("ResourceId", get_json_object(col("RequestParams"), "$.path"))
 .withColumn("ResourceType", lit("DBFS_Path"))
 .withColumn("CapabilityId", lit("CAN_WRITE_TO_DBFS"))
 .select("UserEmail", "ResourceId", "ResourceType", "CapabilityId", 
 "ActionName", "TimeGenerated", "SourceIPAddress")
)

# Parse DatabricksClusters
clusters_df = (raw_clusters
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("ActionName").isin(["create", "edit", "installLibrary"]))
 .transform(parse_common_fields)
 .withColumn("ResourceId", coalesce(
 get_json_object(col("RequestParams"), "$.cluster_id"),
 get_json_object(col("RequestParams"), "$.cluster_name")))
 .withColumn("ResourceType", lit("Cluster"))
 .withColumn("CapabilityId", lit("CAN_CONTROL_CLUSTER"))
 .select("UserEmail", "ResourceId", "ResourceType", "CapabilityId", 
 "ActionName", "TimeGenerated", "SourceIPAddress")
)

# Parse DatabricksJobs
jobs_df = (raw_jobs
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("ActionName").isin(["create", "update", "runNow"]))
 .transform(parse_common_fields)
 .withColumn("ResourceId", coalesce(
 get_json_object(col("RequestParams"), "$.job_id"),
 get_json_object(col("RequestParams"), "$.name")))
 .withColumn("ResourceType", lit("Job"))
 .withColumn("CapabilityId", lit("CAN_SCHEDULE_JOBS"))
 .select("UserEmail", "ResourceId", "ResourceType", "CapabilityId", 
 "ActionName", "TimeGenerated", "SourceIPAddress")
)

# Parse DatabricksSQLPermissions
permissions_df = (raw_permissions
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("ActionName").isin(["grant", "transferOwnership"]))
 .transform(parse_common_fields)
 .withColumn("ResourceId", get_json_object(col("RequestParams"), "$.securable_full_name"))
 .withColumn("ResourceType", lit("SQLObject"))
 .withColumn("CapabilityId", lit("CAN_GRANT_PERMISSIONS"))
 .select("UserEmail", "ResourceId", "ResourceType", "CapabilityId", 
 "ActionName", "TimeGenerated", "SourceIPAddress")
)

# UNION all into single outbound DataFrame
outbound_df = (notebook_df
 .union(secrets_df)
 .union(dbfs_df)
 .union(clusters_df)
 .union(jobs_df)
 .union(permissions_df)
 .where(col("UserEmail").isNotNull())
 .where(col("ResourceId").isNotNull())
).persist()

print(f" Unified outbound events: {outbound_df.count()}")
outbound_df.groupBy("CapabilityId").count().orderBy("count", ascending=False).show()

## Node & Edge DataFrames

We prepare lookup DataFrames from identity and risk tables. These provide identity and risk context:
- **Identity Context**: Department, JobTitle, UserType, Manager (from IdentityInfo)
- **Entra Risk Level**: The risk level assigned by Entra ID itself 
- **UEBA Priority**: The investigation priority from BehaviorAnalytics 

In [ ]:
# =============================================================================
# STEP 2b: Prepare Enrichment Lookup DataFrames
# =============================================================================

# ===================== IDENTITY INFO (from Entra) =====================
identity_enrichment = (raw_identity
 .select(
 lower(trim(col("AccountUpn"))).alias("UserEmailLower"),
 col("Department"),
 col("JobTitle"),
 col("UserType"),
 col("Manager"),
 col("AccountObjectId")
 )
 .dropDuplicates(["UserEmailLower"])
)

print(f" Identity enrichment prepared: {identity_enrichment.count()} users")

# ===================== ENTRA ID RISK EVENTS (from Entra) =====================
risk_enrichment = (raw_risk_events
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .groupBy(lower(trim(col("UserPrincipalName"))).alias("UserEmailLower"))
 .agg(
  first(col("RiskLevel"), ignorenulls=True).alias("EntraRiskLevel"),
 countDistinct("RiskEventType").alias("UniqueRiskEventTypes"),
 count("*").alias("TotalRiskEvents")
 )
)

print(f" Risk events enrichment prepared: {risk_enrichment.count()} users")

# ===================== BEHAVIOR ANALYTICS (from UEBA) =====================
behavior_enrichment = (raw_behavior
 .where(to_date(col("TimeGenerated")) >= LOOKBACK_DATE)
 .where(col("UserPrincipalName").isNotNull())
 .groupBy(lower(trim(col("UserPrincipalName"))).alias("UserEmailLower"))
 .agg(
 max("InvestigationPriority").alias("MaxInvestigationPriority"),
 countDistinct("ActivityType").alias("UniqueAnomalyTypes"),
 count("*").alias("TotalAnomalies")
 )
)

print(f" Behavior analytics enrichment prepared: {behavior_enrichment.count()} users")

In [ ]:
# =============================================================================
# STEP 3: Extract NODE DataFrames
# =============================================================================

# User nodes with activity metrics
user_base_df = (outbound_df
 .groupBy("UserEmail")
 .agg(
 countDistinct("CapabilityId").alias("OutboundPathTypes"), # count of capabilities
 count("*").alias("TotalOutboundEvents"), # total events
 countDistinct("ResourceId").alias("UniqueResources") # unique resources
 )
 .withColumn("UserEmailLower", lower(trim(col("UserEmail"))))
)

# ===================== ENRICH USER NODES =====================
user_enriched_df = user_base_df.join(identity_enrichment, on="UserEmailLower", how="left")
user_enriched_df = user_enriched_df.join(risk_enrichment, on="UserEmailLower", how="left")
user_enriched_df = user_enriched_df.join(behavior_enrichment, on="UserEmailLower", how="left")

# Final user nodes with enrichment
user_nodes_df = (user_enriched_df
 .fillna({
 "Department": "Unknown",
 "JobTitle": "Unknown",
 "UserType": "Unknown",
 "EntraRiskLevel": "none",
 "MaxInvestigationPriority": 0
 })
 .select(
 "UserEmail",
 # Activity counts
 "OutboundPathTypes",
 "TotalOutboundEvents",
 "UniqueResources",
 # Identity context from Entra
 "Department",
 "JobTitle",
 "UserType",
 "Manager",
 # Risk signals from source systems
 "EntraRiskLevel", # From AADUserRiskEvents
 "UniqueRiskEventTypes",
 "TotalRiskEvents",
 "MaxInvestigationPriority", # From BehaviorAnalytics
 "UniqueAnomalyTypes",
 "TotalAnomalies"
 )
)

print(f" User nodes: {user_nodes_df.count()} users")
print(" \n=== Sample Users ===")
user_nodes_df.select("UserEmail", "Department", "OutboundPathTypes", "EntraRiskLevel", "MaxInvestigationPriority").show(5, truncate=False)

# ===================== RESOURCE NODES  =====================
resource_nodes_df = (outbound_df
 .select(
 col("ResourceId"),
 col("ResourceType")
 )
 .distinct()
)

# ===================== CAPABILITY NODES =====================
capability_nodes_df = (outbound_df
 .select(col("CapabilityId"))
 .distinct()
)

print(f" Nodes - Users: {user_nodes_df.count()}, Resources: {resource_nodes_df.count()}, Capabilities: {capability_nodes_df.count()}")

In [ ]:
# =============================================================================
# STEP 4: Extract EDGE DataFrames
# =============================================================================

# User -> Resource edges (raw timestamps, no after-hours flag)
user_resource_edges_df = (outbound_df
 .select(
 col("UserEmail"),
 col("ResourceId"),
 col("CapabilityId").alias("EdgeType"),
 col("ActionName"),
 col("TimeGenerated"),
 col("SourceIPAddress")
 )
 .withColumn("EdgeKey", concat_ws("_", col("UserEmail"), col("ResourceId"), col("TimeGenerated").cast("string")))
)

# Resource -> Capability edges
resource_capability_edges_df = (outbound_df
 .select(
 col("ResourceId"),
 col("CapabilityId")
 )
 .distinct()
 .withColumn("EdgeKey", concat_ws("_", col("ResourceId"), col("CapabilityId")))
)

# User -> Capability edges (aggregated with timestamps)
user_capability_edges_df = (outbound_df
 .groupBy("UserEmail", "CapabilityId")
 .agg(
 count("*").alias("EventCount"),
 min("TimeGenerated").alias("FirstSeen"),
 max("TimeGenerated").alias("LastSeen")
 )
 .withColumn("EdgeKey", concat_ws("_", col("UserEmail"), col("CapabilityId")))
)

print(f" Edges - ACCESSED: {user_resource_edges_df.count()}")
print(f" Edges - ResourceHasCapability: {resource_capability_edges_df.count()}")
print(f" Edges - UserHasCapability: {user_capability_edges_df.count()}")

## Graph Assembly

Building the graph:
- **3 Node Types**: User (with identity context), Resource, Capability
- **3 Edge Types**: ACCESSED, ResourceHasCapability, UserHasCapability



In [ ]:
# =============================================================================
# STEP 5: Build Graph
# =============================================================================

outbound_graph = (GraphSpecBuilder.start()
 
 # ===================== NODES (3 types) =====================
 .add_node("User")
 .from_dataframe(user_nodes_df)
 .with_columns(
 # Activity counts
 "UserEmail", "OutboundPathTypes", "TotalOutboundEvents", "UniqueResources",
 # Identity context
 "Department", "JobTitle", "UserType", "Manager",
 # Risk signals from source systems
 "EntraRiskLevel", "UniqueRiskEventTypes", "TotalRiskEvents",
 "MaxInvestigationPriority", "UniqueAnomalyTypes", "TotalAnomalies",
 key="UserEmail", display="UserEmail"
 )

 .add_node("Resource")
 .from_dataframe(resource_nodes_df)
 .with_columns("ResourceId", "ResourceType",
 key="ResourceId", display="ResourceId")

 .add_node("Capability")
 .from_dataframe(capability_nodes_df)
 .with_columns("CapabilityId",
 key="CapabilityId", display="CapabilityId")

 # ===================== EDGES (3 types) =====================
 .add_edge("ACCESSED")
 .from_dataframe(user_resource_edges_df)
 .source(id_column="UserEmail", node_type="User")
 .target(id_column="ResourceId", node_type="Resource")
 .with_columns("EdgeType", "ActionName", "TimeGenerated", "SourceIPAddress", "EdgeKey",
 key="EdgeKey", display="EdgeKey")

 .add_edge("ResourceHasCapability")
 .from_dataframe(resource_capability_edges_df)
 .source(id_column="ResourceId", node_type="Resource")
 .target(id_column="CapabilityId", node_type="Capability")
 .with_columns("EdgeKey",
 key="EdgeKey", display="EdgeKey")

 .add_edge("UserHasCapability")
 .from_dataframe(user_capability_edges_df)
 .source(id_column="UserEmail", node_type="User")
 .target(id_column="CapabilityId", node_type="Capability")
 .with_columns("EventCount", "FirstSeen", "LastSeen", "EdgeKey",
 key="EdgeKey", display="EdgeKey")

).done()



In [ ]:
# Check the schema of the graph spec to ensure it is correct
outbound_graph.show_schema()


## Build & Publish

Build the graph from the spec. This validates the schema, prepares the data, and publishes the graph for querying.


In [ ]:
# Build the graph from the spec - this will validate the spec and prepare it for querying
# Alternative: use Graph.prepare() to prepare the graph nodes and edges in the lake
# and then use Graph.publish() to create the graph. You would typically call prepare()
# and publish() separately to understand the cost of Graph API calls triggered by Graph.publish()
# see https://learn.microsoft.com/azure/sentinel/billing?tabs=simplified%2Ccommitment-tiers
databricks_graph = Graph.build(outbound_graph)
